# pi-eval on Kaggle (2x T4) - Qwen3-4B via Ollama

Runs the pi-eval harness against **qwen3:4b** served by Ollama at its maximum context window (262,144 tokens). Ollama's OpenAI-compatible endpoint at `http://localhost:11434/v1` is the eval provider (`--api-base`). The suite runs on the host backend inside the Kaggle container.

**Before running:**
1. Kaggle UI: set **Accelerator -> GPU T4 x2**.
2. Kaggle UI: enable **Internet** (apt, npm, pip, Ollama, model pull).
3. No secret is needed: the default `REPO_URL` is the public repo. To use a fork instead, set the Kaggle Secret `PI_EVAL_REPO_URL`. A Kaggle Dataset named `pi-eval` also works.

Setup notes (these fix earlier failures):
- The Ollama library tag is `qwen3:4b`. There is no `qwen3:4b-thinking-2507` tag.
- Kaggle has no systemd, so `ollama serve` is started manually **before** any pull, after installing `zstd`.
- `pi_eval.py` is committed in the repo now. A cell below still writes the exact client into the clone so the notebook also works against older clones.
- If 262,144 tokens OOM the 2x T4 KV cache, lower `MAX_CTX` in the config cell and rerun from the 'Create the maximum-context model' cell.

In [ ]:
!nvidia-smi
!free -h
!nproc
!df -h /kaggle/working

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get('PI_EVAL_REPO_URL', 'https://github.com/velostudioagency/pi-eval.git')
REPO_DIR = Path('/kaggle/working/pi-eval')
DATASET_DIR = Path('/kaggle/input/pi-eval')

BASE_MODEL = 'qwen3:4b'     # real Ollama library tag; ~2.5 GB, 256K native context
MODEL_ID = 'qwen3:4b-max'   # same weights with num_ctx pinned to MAX_CTX
MAX_CTX = 262144            # 256K; lower this if the KV cache OOMs on 2x T4
PORT = 11434
RUNS = 5
SUITE = 'extraction'
print(BASE_MODEL, MODEL_ID, MAX_CTX, SUITE, RUNS)

## Get the code

Clone from `PI_EVAL_REPO_URL`, copy from a Kaggle Dataset, or reuse a directory that is already present.

In [ ]:
import shutil
import subprocess

if REPO_DIR.exists():
    print('repo already present at', REPO_DIR)
elif DATASET_DIR.exists():
    shutil.copytree(DATASET_DIR, REPO_DIR)
    print('copied repo from', DATASET_DIR)
elif REPO_URL and '<your-user>' not in REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    raise ValueError('Set Kaggle Secret PI_EVAL_REPO_URL, or add the repo as a Kaggle Dataset named pi-eval.')
print(sorted(p.name for p in REPO_DIR.iterdir())[:12])

## Install Node 22+ and the repo dependencies

The harness needs Node 22.19+. Model catalog data is gitignored, so hydrate it and build the workspace packages once.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y nodejs
!node --version && npm --version

In [ ]:
%cd /kaggle/working/pi-eval
!npm ci --ignore-scripts
!npm run hydrate:model-data
!npm run build:offline

In [ ]:
!pip -q install requests pandas huggingface_hub

## Install Ollama

The official installer needs `zstd` on Kaggle's Ubuntu image. Install it first.

In [ ]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## Serve the model

Kaggle has no systemd, so the Ollama installer does not start a server. Start `ollama serve` in the background and wait for the API. All `ollama` commands below depend on this server, so this cell must stay before the pull.

In [ ]:
import subprocess
import time

import requests

log = open('/kaggle/working/ollama.log', 'w')
server = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT)
print('ollama serve pid', server.pid)

ready = False
for _ in range(60):
    try:
        if requests.get(f'http://localhost:{PORT}/api/tags', timeout=2).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    server.terminate()
    log.close()
    raise RuntimeError('ollama serve did not become ready; see /kaggle/working/ollama.log')

tags = requests.get(f'http://localhost:{PORT}/api/tags', timeout=2).json()
print('serving:', [m['name'] for m in tags.get('models', [])])

## Pull the base model

Pull `qwen3:4b` (the real Ollama library tag). Never use `qwen3:4b-thinking-2507`; it does not exist.

In [ ]:
subprocess.run(['ollama', 'pull', BASE_MODEL], check=True)
!ollama show {BASE_MODEL} --modelfile | grep -i -E 'FROM|num_ctx'

## Create the maximum-context model

Pin `num_ctx` to 262,144 in a Modelfile and create `qwen3:4b-max`. The eval targets this tag so the model name sent to the OpenAI-compatible endpoint matches the served model. If the KV cache OOMs, lower `MAX_CTX` in the config cell and rerun this cell.

In [ ]:
modelfile = f'FROM {BASE_MODEL}\nPARAMETER num_ctx {MAX_CTX}\n'
with open('/kaggle/working/Modelfile', 'w') as f:
    f.write(modelfile)
print(modelfile)

subprocess.run(['ollama', 'create', MODEL_ID, '-f', '/kaggle/working/Modelfile'], check=True)
!ollama show {MODEL_ID} --modelfile | grep -i -E 'FROM|num_ctx'
!ollama list

## Install the eval client into the clone

`packages/evals/notebooks/pi_eval.py` is now tracked in the repo. The cell below still writes an exact copy into the clone so the notebook also works against older clones; `from pi_eval import PiEval` then succeeds either way.

In [ ]:
import base64

_client_b64 = "IiIiVGhpbiBjbGllbnQgZm9yIHRoZSBQaSBldmFsIHJ1bm5lci4KClVzZSBpdCBmcm9tIGEgbGFwdG9wLCBDb2xhYiwgb3IgS2FnZ2xlLiBUaGUgbW9kZWwgbWF5IHJ1biBhbnl3aGVyZTsgdGhlIHJ1bm5lcgpleGVjdXRlcyB0aGUgYWdlbnQgb24gdGhlIE9TIG5hbWVkIGluIHRoZSBzdWl0ZSdzIGVudmlyb25tZW50IGNvbmZpZy4KCiAgICBmcm9tIHBpX2V2YWwgaW1wb3J0IFBpRXZhbAogICAgY2xpZW50ID0gUGlFdmFsKCJodHRwczovL215LXdpbmRvd3MtcnVubmVyOjg3ODciLCB0b2tlbj1vcy5lbnZpcm9uWyJSVU5ORVJfVE9LRU4iXSkKICAgIHJlc3VsdCA9IGNsaWVudC5ydW4oc3VpdGU9ImV4YW1wbGUiLCBvcz0id2luZG93cyIpCiAgICByZXN1bHQuc2F2ZSgiLi9ydW4iKQogICAgcmVzdWx0LnRyYWNlc19mcmFtZSgpLmhlYWQoKQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgdGVtcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCByZXF1ZXN0cwoKCmNsYXNzIFJ1blJlc3VsdDoKICAgICIiIkFydGlmYWN0cyByZXR1cm5lZCBieSBvbmUgc3VpdGUgcnVuLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXlsb2FkOiBkaWN0W3N0ciwgQW55XSwgYXJ0aWZhY3RfZGlyOiBQYXRoIHwgTm9uZSA9IE5vbmUpOgogICAgICAgIHNlbGYucGF5bG9hZCA9IHBheWxvYWQKICAgICAgICBzZWxmLnJlcG9ydCA9IHBheWxvYWQuZ2V0KCJyZXBvcnQiKQogICAgICAgIHNlbGYudHJhY2VzID0gcGF5bG9hZC5nZXQoInRyYWNlcyIsIFtdKQogICAgICAgIHNlbGYuZGF0YXNldHMgPSBwYXlsb2FkLmdldCgiZGF0YXNldHMiLCB7fSkKICAgICAgICBzZWxmLm9rID0gcGF5bG9hZC5nZXQoIm9rIiwgVHJ1ZSkKICAgICAgICBzZWxmLnN0ZG91dCA9IHBheWxvYWQuZ2V0KCJzdGRvdXQiLCAiIikKICAgICAgICBzZWxmLnN0ZGVyciA9IHBheWxvYWQuZ2V0KCJzdGRlcnIiLCAiIikKCiAgICBkZWYgc2F2ZShzZWxmLCBwYXRoOiBzdHIgfCBQYXRoKSAtPiBQYXRoOgogICAgICAgICIiIldyaXRlIHJlcG9ydCwgdHJhY2VzLCBhbmQgZGF0YXNldHMgdG8gYSBsb2NhbCBkaXJlY3RvcnkuIiIiCiAgICAgICAgdGFyZ2V0ID0gUGF0aChwYXRoKQogICAgICAgIHRhcmdldC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgaWYgc2VsZi5yZXBvcnQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICh0YXJnZXQgLyAicmVwb3J0Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc2VsZi5yZXBvcnQsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICB3aXRoICh0YXJnZXQgLyAidHJhY2VzLmpzb25sIikub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICAgICAgZm9yIHRyYWNlIGluIHNlbGYudHJhY2VzOgogICAgICAgICAgICAgICAgaGFuZGxlLndyaXRlKGpzb24uZHVtcHModHJhY2UpICsgIlxuIikKICAgICAgICBmb3IgbmFtZSwgbGluZXMgaW4gc2VsZi5kYXRhc2V0cy5pdGVtcygpOgogICAgICAgICAgICB3aXRoICh0YXJnZXQgLyBuYW1lKS5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgICAgICAgICAgZm9yIGxpbmUgaW4gbGluZXM6CiAgICAgICAgICAgICAgICAgICAgaGFuZGxlLndyaXRlKGpzb24uZHVtcHMobGluZSkgKyAiXG4iKQogICAgICAgIHJldHVybiB0YXJnZXQKCiAgICBkZWYgdHJhY2VzX2ZyYW1lKHNlbGYpOgogICAgICAgICIiIlJldHVybiB0cmFjZXMgYXMgYSBwYW5kYXMgRGF0YUZyYW1lIChvbmUgcm93IHBlciB0dXJuKS4iIiIKICAgICAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCgogICAgICAgIHJvd3MgPSBbXQogICAgICAgIGZvciB0cmFjZSBpbiBzZWxmLnRyYWNlczoKICAgICAgICAgICAgbWV0YWRhdGEgPSB0cmFjZS5nZXQoIm1ldGFkYXRhIiwge30pCiAgICAgICAgICAgIGZvciBpbmRleCwgbWVzc2FnZSBpbiBlbnVtZXJhdGUodHJhY2UuZ2V0KCJtZXNzYWdlcyIsIFtdKSk6CiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZCh7InJ1biI6IG1ldGFkYXRhLmdldCgiY2FzZUlkIiksICJpbmRleCI6IGluZGV4LCAqKm1lc3NhZ2V9KQogICAgICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykKCgpjbGFzcyBQaUV2YWw6CiAgICAiIiJEcml2ZSB0aGUgZXZhbCBydW5uZXIgb3ZlciBIVFRQLCBvciB0aGUgbG9jYWwgQ0xJIHdoZW4gbm8gVVJMIGlzIGdpdmVuLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHJ1bm5lcl91cmw6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgICAgIHRva2VuOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICByZXBvX3BhdGg6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKICAgICAgICB0aW1lb3V0OiBpbnQgPSAxODAwLAogICAgKToKICAgICAgICBzZWxmLnJ1bm5lcl91cmwgPSBydW5uZXJfdXJsLnJzdHJpcCgiLyIpIGlmIHJ1bm5lcl91cmwgZWxzZSBOb25lCiAgICAgICAgc2VsZi50b2tlbiA9IHRva2VuIG9yIG9zLmVudmlyb24uZ2V0KCJQSV9FVkFMX1JVTk5FUl9UT0tFTiIpCiAgICAgICAgc2VsZi5yZXBvX3BhdGggPSBQYXRoKHJlcG9fcGF0aCkgaWYgcmVwb19wYXRoIGVsc2UgTm9uZQogICAgICAgIHNlbGYudGltZW91dCA9IHRpbWVvdXQKCiAgICBkZWYgaGVhbHRoKHNlbGYpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgIGlmIG5vdCBzZWxmLnJ1bm5lcl91cmw6CiAgICAgICAgICAgIHJldHVybiB7Im9rIjogc2VsZi5yZXBvX3BhdGggaXMgbm90IE5vbmUsICJtb2RlIjogImxvY2FsIn0KICAgICAgICByZXNwb25zZSA9IHJlcXVlc3RzLmdldChmIntzZWxmLnJ1bm5lcl91cmx9L2hlYWx0aCIsIHRpbWVvdXQ9MzApCiAgICAgICAgcmVzcG9uc2UucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgcmV0dXJuIHJlc3BvbnNlLmpzb24oKQoKICAgIGRlZiBydW4oCiAgICAgICAgc2VsZiwKICAgICAgICBzdWl0ZTogc3RyID0gImV4YW1wbGUiLAogICAgICAgIHByb3ZpZGVyOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBtb2RlbDogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgcHJvZmlsZXM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLAogICAgICAgIHRhc2tzOiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSwKICAgICAgICBmYW1pbGllczogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgZG9tYWluczogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgcnVuczogaW50IHwgTm9uZSA9IE5vbmUsCiAgICAgICAgYmFja2VuZDogc3RyID0gImhvc3QiLAogICAgICAgIG9zX25hbWU6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgICAgIGFwaV9iYXNlOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBhcGlfa2V5OiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBhcGk6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgICAgIGV4cG9ydHM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBSdW5SZXN1bHQ6CiAgICAgICAgIiIiUnVuIGEgc3VpdGUgYW5kIHJldHVybiBpdHMgcmVwb3J0LCB0cmFjZXMsIGFuZCBkYXRhc2V0cy4iIiIKICAgICAgICBwYXlsb2FkOiBkaWN0W3N0ciwgQW55XSA9IHsic3VpdGUiOiBzdWl0ZSwgImJhY2tlbmQiOiBiYWNrZW5kfQogICAgICAgIGZvciBrZXksIHZhbHVlIGluIHsKICAgICAgICAgICAgInByb3ZpZGVyIjogcHJvdmlkZXIsCiAgICAgICAgICAgICJtb2RlbCI6IG1vZGVsLAogICAgICAgICAgICAicHJvZmlsZXMiOiBwcm9maWxlcywKICAgICAgICAgICAgInRhc2tzIjogdGFza3MsCiAgICAgICAgICAgICJmYW1pbGllcyI6IGZhbWlsaWVzLAogICAgICAgICAgICAiZG9tYWlucyI6IGRvbWFpbnMsCiAgICAgICAgICAgICJydW5zIjogcnVucywKICAgICAgICAgICAgIm9zIjogb3NfbmFtZSwKICAgICAgICAgICAgImFwaUJhc2UiOiBhcGlfYmFzZSwKICAgICAgICAgICAgImFwaUtleSI6IGFwaV9rZXksCiAgICAgICAgICAgICJhcGkiOiBhcGksCiAgICAgICAgICAgICJleHBvcnRzIjogZXhwb3J0cywKICAgICAgICB9Lml0ZW1zKCk6CiAgICAgICAgICAgIGlmIHZhbHVlIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcGF5bG9hZFtrZXldID0gdmFsdWUKCiAgICAgICAgaWYgc2VsZi5ydW5uZXJfdXJsOgogICAgICAgICAgICBoZWFkZXJzID0geyJjb250ZW50LXR5cGUiOiAiYXBwbGljYXRpb24vanNvbiJ9CiAgICAgICAgICAgIGlmIHNlbGYudG9rZW46CiAgICAgICAgICAgICAgICBoZWFkZXJzWyJhdXRob3JpemF0aW9uIl0gPSBmIkJlYXJlciB7c2VsZi50b2tlbn0iCiAgICAgICAgICAgIHJlc3BvbnNlID0gcmVxdWVzdHMucG9zdChmIntzZWxmLnJ1bm5lcl91cmx9L3J1bnMiLCBqc29uPXBheWxvYWQsIGhlYWRlcnM9aGVhZGVycywgdGltZW91dD1zZWxmLnRpbWVvdXQpCiAgICAgICAgICAgIHJlc3BvbnNlLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgICAgICAgICByZXR1cm4gUnVuUmVzdWx0KHJlc3BvbnNlLmpzb24oKSkKCiAgICAgICAgaWYgbm90IHNlbGYucmVwb19wYXRoOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJMb2NhbCBtb2RlIG5lZWRzIHJlcG9fcGF0aCwgb3IgdXNlIGEgcnVubmVyX3VybC4iKQogICAgICAgIGFydGlmYWN0X2RpciA9IFBhdGgodGVtcGZpbGUubWtkdGVtcChwcmVmaXg9InBpLWV2YWwtbm90ZWJvb2stIikpCiAgICAgICAgY29tbWFuZCA9IFsKICAgICAgICAgICAgIm5wbSIsCiAgICAgICAgICAgICJydW4iLAogICAgICAgICAgICAiZXZhbDpzdWl0ZSIsCiAgICAgICAgICAgICItdyIsCiAgICAgICAgICAgICJwYWNrYWdlcy9ldmFscyIsCiAgICAgICAgICAgICItLSIsCiAgICAgICAgICAgICItLXN1aXRlIiwKICAgICAgICAgICAgc3VpdGUsCiAgICAgICAgICAgICItLWJhY2tlbmQiLAogICAgICAgICAgICBiYWNrZW5kLAogICAgICAgICAgICAiLS1hcnRpZmFjdC1kaXIiLAogICAgICAgICAgICBzdHIoYXJ0aWZhY3RfZGlyKSwKICAgICAgICBdCiAgICAgICAgaWYgcHJvdmlkZXI6CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLXByb3ZpZGVyIiwgcHJvdmlkZXJdCiAgICAgICAgaWYgbW9kZWw6CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLW1vZGVsIiwgbW9kZWxdCiAgICAgICAgaWYgcHJvZmlsZXM6CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLXByb2ZpbGUiLCAiLCIuam9pbihwcm9maWxlcyldCiAgICAgICAgaWYgdGFza3M6CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLXRhc2siLCAiLCIuam9pbih0YXNrcyldCiAgICAgICAgaWYgZmFtaWxpZXM6CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLWZhbWlseSIsICIsIi5qb2luKGZhbWlsaWVzKV0KICAgICAgICBpZiBkb21haW5zOgogICAgICAgICAgICBjb21tYW5kICs9IFsiLS1kb21haW4iLCAiLCIuam9pbihkb21haW5zKV0KICAgICAgICBpZiBydW5zIGlzIG5vdCBOb25lOgogICAgICAgICAgICBjb21tYW5kICs9IFsiLS1ydW5zIiwgc3RyKHJ1bnMpXQogICAgICAgIGlmIG9zX25hbWU6CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLW9zIiwgb3NfbmFtZV0KICAgICAgICBpZiBhcGlfYmFzZToKICAgICAgICAgICAgY29tbWFuZCArPSBbIi0tYXBpLWJhc2UiLCBhcGlfYmFzZV0KICAgICAgICBpZiBhcGlfa2V5OgogICAgICAgICAgICBjb21tYW5kICs9IFsiLS1hcGkta2V5IiwgYXBpX2tleV0KICAgICAgICBpZiBhcGk6CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLWFwaSIsIGFwaV0KICAgICAgICBmb3IgZXhwb3J0IGluIGV4cG9ydHMgb3IgW106CiAgICAgICAgICAgIGNvbW1hbmQgKz0gWyItLWV4cG9ydCIsIGV4cG9ydF0KICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bihjb21tYW5kLCBjd2Q9c2VsZi5yZXBvX3BhdGgsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD1zZWxmLnRpbWVvdXQpCiAgICAgICAgcmV0dXJuIFJ1blJlc3VsdCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgIm9rIjogcmVzdWx0LnJldHVybmNvZGUgaW4gKDAsIDEpLAogICAgICAgICAgICAgICAgImV4aXRDb2RlIjogcmVzdWx0LnJldHVybmNvZGUsCiAgICAgICAgICAgICAgICAicmVwb3J0IjoganNvbi5sb2FkcygoYXJ0aWZhY3RfZGlyIC8gInJlcG9ydC5qc29uIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgICAgICAgICAgaWYgKGFydGlmYWN0X2RpciAvICJyZXBvcnQuanNvbiIpLmV4aXN0cygpCiAgICAgICAgICAgICAgICBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAidHJhY2VzIjogW2pzb24ubG9hZHMobGluZSkgZm9yIGxpbmUgaW4gKGFydGlmYWN0X2RpciAvICJ0cmFjZXMuanNvbmwiKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpIGlmIGxpbmVdCiAgICAgICAgICAgICAgICBpZiAoYXJ0aWZhY3RfZGlyIC8gInRyYWNlcy5qc29ubCIpLmV4aXN0cygpCiAgICAgICAgICAgICAgICBlbHNlIFtdLAogICAgICAgICAgICAgICAgImRhdGFzZXRzIjogewogICAgICAgICAgICAgICAgICAgIHBhdGgubmFtZTogW2pzb24ubG9hZHMobGluZSkgZm9yIGxpbmUgaW4gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpIGlmIGxpbmVdCiAgICAgICAgICAgICAgICAgICAgZm9yIHBhdGggaW4gYXJ0aWZhY3RfZGlyLmdsb2IoImRhdGFzZXQuKi5qc29ubCIpCiAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgInN0ZG91dCI6IHJlc3VsdC5zdGRvdXRbLTQwMDA6XSwKICAgICAgICAgICAgICAgICJzdGRlcnIiOiByZXN1bHQuc3RkZXJyWy00MDAwOl0sCiAgICAgICAgICAgIH0KICAgICAgICApCg=="
client_path = REPO_DIR / 'packages' / 'evals' / 'notebooks' / 'pi_eval.py'
client_path.write_bytes(base64.b64decode(_client_b64))
print('wrote', client_path, client_path.stat().st_size, 'bytes')

## Run the eval

First a one-task smoke run to prove the endpoint works, then the full suite with repetitions. Both use the `host` backend and the Ollama OpenAI-compatible endpoint.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR / 'packages' / 'evals' / 'notebooks'))
from pi_eval import PiEval

client = PiEval(repo_path=REPO_DIR, timeout=7200)
smoke = client.run(
    suite=SUITE,
    tasks=['oven-payment-real'],
    runs=1,
    backend='host',
    model=MODEL_ID,
    api_base=f'http://localhost:{PORT}/v1',
)
print('ok:', smoke.ok, 'traces:', len(smoke.traces))
if not smoke.ok:
    print(smoke.stderr[-2000:])
smoke.report

## Run the full suite with a live transcript

The harness buffers the inner eval process, so it cannot stream tokens. It does rewrite `traces.jsonl` after every completed task, and each trace contains assistant text, `reasoning`, `toolCalls` (name and arguments), and tool results. The cells below launch the CLI in the background and poll that file, so each finished run appears as it completes. Re-run the viewer cell if you interrupt it.

In [ ]:
import subprocess, json, time
from pathlib import Path
from IPython.display import clear_output

LIVE_DIR = Path('/kaggle/working/pi-eval-live')
LIVE_DIR.mkdir(parents=True, exist_ok=True)
cli_log = open(LIVE_DIR / 'cli.log', 'w')

cmd = [
    'npm', 'run', 'eval:suite', '-w', 'packages/evals', '--',
    '--suite', SUITE,
    '--backend', 'host',
    '--runs', str(RUNS),
    '--model', MODEL_ID,
    '--api-base', f'http://localhost:{PORT}/v1',
    '--artifact-dir', str(LIVE_DIR),
]
live = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=cli_log, stderr=subprocess.STDOUT, text=True)
print('started pid', live.pid, '->', LIVE_DIR)

In [ ]:
expected_path = LIVE_DIR / 'expected-runs.json'
TOTAL_RUNS = len(json.loads(expected_path.read_text(encoding='utf-8'))) if expected_path.exists() else RUNS * 4
seen = 0
transcript = []


def collect_new():
    global seen
    path = LIVE_DIR / 'traces.jsonl'
    if not path.exists():
        return
    traces = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]
    for trace in traces[seen:]:
        meta = trace.get('metadata', {})
        parts = [f"\n=== {meta.get('caseId')} | {meta.get('profile')} | run {meta.get('runNumber')} | score {meta.get('score')} ==="]
        for msg in trace.get('messages', []):
            role = msg.get('role')
            if role == 'system':
                continue
            if role == 'user':
                parts.append('\n[USER] ' + str(msg.get('content'))[:600])
            elif role == 'assistant':
                if msg.get('reasoning'):
                    parts.append('\n[REASONING] ' + msg['reasoning'][:2000])
                if msg.get('content'):
                    parts.append('\n[ASSISTANT] ' + msg['content'][:2000])
                for call in msg.get('toolCalls', []):
                    parts.append('\n[TOOL CALL] ' + str(call.get('name')) + ' ' + json.dumps(call.get('arguments'))[:800])
            elif role == 'tool':
                flag = ' (ERROR)' if msg.get('isError') else ''
                parts.append('\n[TOOL RESULT] ' + str(msg.get('name')) + flag + ' ' + str(msg.get('content'))[:1500])
        transcript.append('\n'.join(parts))
    seen = len(traces)


while True:
    collect_new()
    tail = (LIVE_DIR / 'cli.log').read_text(encoding='utf-8')[-1500:] if (LIVE_DIR / 'cli.log').exists() else ''
    clear_output(wait=True)
    print(f'=== live eval | pid {live.pid} | running {live.poll() is None} | traces {seen}/{TOTAL_RUNS} ===')
    print('\n'.join(transcript)[-12000:])
    print('\n--- CLI log tail ---\n' + tail)
    if live.poll() is not None:
        break
    time.sleep(10)

cli_log.close()
print('\nFINISHED with exit code', live.returncode)

In [ ]:
report = json.loads((LIVE_DIR / 'report.json').read_text(encoding='utf-8'))
print((LIVE_DIR / 'report.txt').read_text(encoding='utf-8'))
print('\nartifacts:', sorted(p.name for p in LIVE_DIR.iterdir()))

## Push the evaluated model to HuggingFace (optional)

Ollama does not push to HuggingFace, so this exports the GGUF blob behind `qwen3:4b-max` and uploads it. It is skipped unless Kaggle Secrets `HF_REPO` and `HF_TOKEN` are set, so it never blocks the eval results above.

In [ ]:
import glob
import json
import shutil

from huggingface_hub import HfApi, upload_file

HF_REPO = os.environ.get('HF_REPO', '')
token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if not HF_REPO or not token:
    print('Skipping HuggingFace push: set Kaggle Secrets HF_REPO and HF_TOKEN to enable.')
else:
    root = os.path.expanduser('~/.ollama/models')
    repo, _, tag = MODEL_ID.partition(':')
    manifests = glob.glob(os.path.join(root, 'manifests', '**', repo, tag), recursive=True)
    layers = [
        layer
        for path in manifests
        for layer in json.load(open(path)).get('layers', [])
        if layer.get('mediaType') == 'application/vnd.ollama.image.model'
    ]
    if not layers:
        raise RuntimeError('no GGUF layer for ' + MODEL_ID)

    blob = os.path.join(root, 'blobs', layers[0]['digest'].replace(':', '-'))
    out = '/kaggle/working/Qwen3-4B-Max.gguf'
    shutil.copy(blob, out)
    print('exported', layers[0].get('size'), 'bytes to', out)

    api = HfApi()
    api.create_repo(HF_REPO, exist_ok=True, private=False)
    upload_file(path_or_fileobj=out, path_in_repo='Qwen3-4B-Max.gguf', repo_id=HF_REPO, token=token)
    print('pushed to https://huggingface.co/' + HF_REPO)

Other suites you can switch to in the config cell above: `docs`, `example`, `data-manipulation-read-organize`, `extraction-minimal`.

In [ ]:
# Stop the model server when finished
server.terminate()
log.close()